In [0]:
-- 0.1 
WITH estadisticas_por_zona AS
(
    SELECT 
        pickup_zip,
        round(avg(fare_amount),2) as avg_fare, 
        round(avg(trip_distance),2) as avg_distance,
        count(*) as cantidad_viajes
    from samples.nyctaxi.trips
    where 
        pickup_zip is not null
        and fare_amount >0
        and trip_distance >0
    group by pickup_zip
    HAVING cantidad_viajes > 100
)
SELECT * from estadisticas_por_zona
order by pickup_zip, avg_fare desc 
limit 100
;

In [0]:
--0.2 
WITH promedio_por_hora AS
(
    SELECT 
        avg(fare_amount) as avg_fare,
        HOUR(tpep_pickup_datetime) as pickup_date
    FROM samples.nyctaxi.trips
    GROUP BY HOUR(tpep_pickup_datetime)
),
promedio_general AS
(
    SELECT avg(fare_amount) as avg_fare_total 
    FROM samples.nyctaxi.trips
)
SELECT ph.pickup_date, round(ph.avg_fare,2) as tarifa_por_hora, round(pg.avg_fare_total,2) as tarifa_general, round(pg.avg_fare_total - ph.avg_fare,2) as diferencia
FROM promedio_por_hora ph
CROSS JOIN promedio_general pg
order by pickup_date asc 







In [0]:
--0.3
WITH viajes_validos AS 
(
    SELECT 
        trip_distance
    FROM samples.nyctaxi.trips
    WHERE 
        trip_distance > 0
        AND fare_amount >0
)
SELECT MAX(trip_distance) as minimimo, MIN(trip_distance) as maximo, ROUND(AVG(trip_distance),2) as promedio FROM viajes_validos


In [0]:
--0.4
SELECT pickup_zip, fare_amount, tpep_pickup_datetime, trip_distance, row_number() OVER(order by fare_amount desc) as rank
FROM samples.nyctaxi.trips
order by rank asc


In [0]:
--0.5
SELECT 
    pickup_zip, 
    fare_amount,
    tpep_pickup_datetime,
    trip_distance,
    row_number() OVER(ORDER BY fare_amount desc) as Select_row,
    RANK() OVER(ORDER BY fare_amount desc) as Rank,
    DENSE_RANK() OVER(ORDER BY fare_amount desc) as Dense_Rank
FROM samples.nyctaxi.trips


In [0]:
--0.6
SELECT 
    pickup_zip, 
    fare_amount,
    trip_distance,
    round(avg(fare_amount) over(),2) as avg_fare
FROM samples.nyctaxi.trips

In [0]:
--0.7
SELECT 
    pickup_zip, 
    tpep_pickup_datetime,
    fare_amount,
    trip_distance,
    round(avg(fare_amount) over(PARTITION BY pickup_zip),2) as avg_fare
FROM samples.nyctaxi.trips
order by pickup_zip


In [0]:
--0.8
SELECT 
    pickup_zip, 
    tpep_pickup_datetime,
    trip_distance,
    fare_amount,
    lag(fare_amount) over(order by tpep_pickup_datetime)
FROM samples.nyctaxi.trips
order by tpep_pickup_datetime

In [0]:
--0.9
SELECT 
    pickup_zip,
    tpep_pickup_datetime,
    fare_amount,
    sum(fare_amount) over(order by tpep_pickup_datetime)
FROM samples.nyctaxi.trips

In [0]:
--0.10
with viajes_validos as (
    select 
        pickup_zip,
        tpep_pickup_datetime,
        trip_distance,
        fare_amount
    from samples.nyctaxi.trips
    where 
        trip_distance > 0
        and fare_amount > 0
)
, distancias as (
    select 
        pickup_zip,
        avg(trip_distance) as avg_trip, max(trip_distance) as max_trip, min(trip_distance) as min_trip,
        avg(fare_amount) as avg_amount, max(fare_amount) as max_amount, min(fare_amount) as min_amount
    from viajes_validos
    group by pickup_zip
)

select pickup_zip, round(avg_amount,2) as avg_amount, max_amount, min_amount, row_number() over(order by avg_amount desc) as rank
from distancias
limit 10
        

In [0]:
-- 1.1
SELECT count(*) FROM bootcamp.bronze.properties_bronze


In [0]:
desc table bootcamp.bronze.properties_bronze

In [0]:
--1.3
select 
    id, ubicacion, precio, expensas, tipo_de_operacion, moneda, ambientes, metros_cuadrados_totales, antiguedad, estado, zona
from bootcamp.bronze.properties_bronze
limit 10

In [0]:
--2.1
WITH total AS (
    SELECT COUNT(*) as n 
    FROM bootcamp.bronze.properties_bronze
),
nulos AS (
    SELECT
        COUNT(*) - COUNT(id) as nulos_id,
        COUNT(*) - COUNT(ubicacion) as nulos_ubicacion,
        COUNT(*) - COUNT(precio) as nulos_precio,
        COUNT(*) - COUNT(expensas) as nulos_expensas,
        COUNT(*) - COUNT(tipo_de_operacion) as nulos_tipo_operacion,
        COUNT(*) - COUNT(moneda) as nulos_moneda,
        COUNT(*) - COUNT(ambientes) as nulos_ambientes,
        COUNT(*) - COUNT(metros_cuadrados_totales) as nulos_m2_totales,
        COUNT(*) - COUNT(metros_cuadrados_cubiertos) as nulos_m2_cubiertos,
        COUNT(*) - COUNT(orientacion_cardinal) as nulos_orientacion_cardinal,
        COUNT(*) - COUNT(orientacion_inmueble) as nulos_orientacion_inmueble,
        COUNT(*) - COUNT(piso) as nulos_piso,
        COUNT(*) - COUNT(cochera) as nulos_cochera,
        COUNT(*) - COUNT(antiguedad) as nulos_antiguedad,
        COUNT(*) - COUNT(estado) as nulos_estado,
        COUNT(*) - COUNT(zona) as nulos_zona
    FROM bootcamp.bronze.properties_bronze
)
SELECT 
    t.n as total_registros,
    n.*
FROM total t, nulos n;

In [0]:
-- 2.3

WITH totales AS (
    SELECT COUNT(*) as total FROM bootcamp.bronze.properties_bronze
)
SELECT 
    ROUND((t.total - COUNT(precio)) * 100.0 / t.total, 2) as pct_nulos_precio,
    ROUND((t.total - COUNT(expensas)) * 100.0 / t.total, 2) as pct_nulos_expensas,
    ROUND((t.total - COUNT(ambientes)) * 100.0 / t.total, 2) as pct_nulos_ambientes,
    ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as pct_nulos_m2_totales,
    ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as pct_nulos_m2_cubiertos,
    ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as pct_nulos_orientacion,
    ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as pct_nulos_antiguedad
FROM bootcamp.bronze.properties_bronze
CROSS JOIN totales t
GROUP BY t.total;

In [0]:
%sql
-- ============================================================
-- EDA 2.3: Identificar columnas con >50% de valores nulos
-- ============================================================
-- Columnas con >50% nulos son críticas: hay que decidir si eliminarlas o imputarlas

WITH totales AS (
    SELECT COUNT(*) as total FROM bootcamp.bronze.properties_bronze
),
pct_nulos AS (
    SELECT 
        ROUND((t.total - COUNT(precio)) * 100.0 / t.total, 2) as precio,
        ROUND((t.total - COUNT(expensas)) * 100.0 / t.total, 2) as expensas,
        ROUND((t.total - COUNT(ambientes)) * 100.0 / t.total, 2) as ambientes,
        ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as metros_cuadrados_totales,
        ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as metros_cuadrados_cubiertos,
        ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as orientacion_cardinal,
        ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as antiguedad,
        ROUND((t.total - COUNT(piso)) * 100.0 / t.total, 2) as piso,
        ROUND((t.total - COUNT(cochera)) * 100.0 / t.total, 2) as cochera
    FROM bootcamp.bronze.properties_bronze
    CROSS JOIN totales t
    GROUP BY t.total
)

-- SELECT * from pct_nulos
SELECT columna, porcentaje_nulos
FROM pct_nulos
UNPIVOT (
    porcentaje_nulos FOR columna IN (
        precio, expensas, ambientes, 
        metros_cuadrados_totales, metros_cuadrados_cubiertos,
        orientacion_cardinal, antiguedad, piso, cochera
    )
)
WHERE porcentaje_nulos > 50
ORDER BY porcentaje_nulos DESC

In [0]:
--3.1
SELECT tipo_de_operacion, count(*) as num_registros, sum(count(*)) over() as total
FROM bootcamp.bronze.properties_bronze
group by tipo_de_operacion
order by num_registros desc


In [0]:
--3.2
SELECT 
    moneda, count(*) as num_registros, sum(count(*)) over() as total, round(count(*)/sum(count(*)) over() * 100,2) as porcentaje
FROM bootcamp.bronze.properties_bronze
group by moneda

In [0]:
--3.3
select ambientes, count(*) as num_registros, sum(count(*)) over() as total, round(count(*)/sum(count(*)) over() * 100,2) as porcentaje
from bootcamp.bronze.properties_bronze
group by ambientes
order by ambientes asc


In [0]:
--3.4
select zona, count(*) as num_registros, sum(count(*)) over() as total, round(count(*)/sum(count(*)) over() * 100,2) as porcentaje
from bootcamp.bronze.properties_bronze
group by zona
order by porcentaje desc 
limit 15

In [0]:
--3.5
select estado, count(*) as num_registros, sum(count(*)) over() as total, round(count(*)/sum(count(*)) over() * 100,2) as porcentaje
from bootcamp.bronze.properties_bronze
group by estado
order by porcentaje desc 

In [0]:
--4.1
select 
    moneda,
    -- tipo_de_operacion,
    count(*),
    min(precio),
    max(precio),
    avg(precio),
    percentile(precio, 0.25),
    percentile(precio, 0.75),
    count(*)
from bootcamp.bronze.properties_bronze
group by moneda


In [0]:
--4.2
select
    'metros_cuadrados_totales' as metros_cuadrados_totales,
    count(*) as cantidad_no_nulos,
    min(metros_cuadrados_totales)::double as minimo_mt_totales,
    max(metros_cuadrados_totales)::double as maximo_mt_totales,
    avg(metros_cuadrados_totales)::double as promedio_mt_totales,
    percentile(metros_cuadrados_totales, 0.5)::double as med_mt_totales,
    'metros_cuadrados_cubiertos' as metros_cuadrados_cubiertos,
    count(*) as cantidad_no_nulos,
    min(metros_cuadrados_cubiertos)::double as minimo_mt_cubiertos,
    max(metros_cuadrados_cubiertos)::double as maximo_mt_cubiertos,
    avg(metros_cuadrados_cubiertos)::double as promedio_mt_cubiertos,
    percentile(metros_cuadrados_cubiertos, 0.5)::double as med_mt_cubiertos
    
from bootcamp.bronze.properties_bronze
where metros_cuadrados_cubiertos is not null and metros_cuadrados_cubiertos::double > 0
and metros_cuadrados_totales is not null and metros_cuadrados_totales::double > 0


In [0]:
--4.3
select antiguedad, count(*) as num_registros
from bootcamp.bronze.properties_bronze
group by antiguedad
order by num_registros desc

In [0]:
--4.4
select try_cast(antiguedad as double) as antiguedad, count(*) as num_registros
from bootcamp.bronze.properties_bronze
where antiguedad::double != '999'
and antiguedad::double is not null 
and antiguedad::double >= 0
group by try_cast(antiguedad as double)
order by num_registros desc

In [0]:
%sql
-- ============================================================
-- EDA 5.1: Reporte de calidad de datos usando CTEs
-- ============================================================

WITH total AS (
    SELECT COUNT(*) as n 
    FROM bootcamp.bronze.properties_bronze
),
problemas AS (
    SELECT
        -- Precios problemáticos
        COUNT(CASE WHEN precio IS NULL OR precio::float <= 0 THEN 1 END) as precio_invalido,
        -- Metros cuadrados problemáticos  
        COUNT(CASE WHEN metros_cuadrados_totales::float IS NULL OR try_cast(metros_cuadrados_totales AS float) <= 0 THEN 1 END) as m2_invalido,
        -- Antigüedad con valor 999 (placeholder)
        COUNT(CASE WHEN antiguedad = '999' OR antiguedad::float = 999 THEN 1 END) as antiguedad_999,
        -- Ambientes = 0 o NUL  L
        COUNT(CASE WHEN ambientes IS NULL OR try_cast(ambientes AS float) = 0 THEN 1 END) as ambientes_invalido,
        -- Moneda vacía
        COUNT(CASE WHEN moneda IS NULL OR moneda = '' THEN 1 END) as moneda_vacia,
        -- Tipo de operación vacía
        COUNT(CASE WHEN tipo_de_operacion IS NULL OR tipo_de_operacion = '' THEN 1 END) as tipo_operacion_vacia
    FROM bootcamp.bronze.properties_bronze
)
SELECT 
    t.n as total_registros,
    p.precio_invalido,
    ROUND(p.precio_invalido * 100.0 / t.n, 2) as pct_precio_invalido,
    p.m2_invalido,
    ROUND(p.m2_invalido * 100.0 / t.n, 2) as pct_m2_invalido,
    p.antiguedad_999,
    ROUND(p.antiguedad_999 * 100.0 / t.n, 2) as pct_antiguedad_999,
    p.ambientes_invalido,
    ROUND(p.ambientes_invalido * 100.0 / t.n, 2) as pct_ambientes_invalido
FROM total t, problemas p;


In [0]:
--5.2 5.3
select
    precio,
    url,
    count(*)
from bootcamp.bronze.properties_bronze
group by precio, url
having count(*) > 1
order by count(*) desc

In [0]:
--5.4
with rango_precios as 
(
select moneda, percentile(precio, 0.01) as pct_001, percentile(precio, 0.99) as pct_99
from bootcamp.bronze.properties_bronze
group by moneda
)
, propiedades_fuera as
(
    select precio, p.moneda
    from bootcamp.bronze.properties_bronze p
    join rango_precios r
    on p.moneda = r.moneda
    where p.precio < r.pct_001 or p.precio > r.pct_99
)
select * from propiedades_fuera

In [0]:
--6.1
select zona, round(avg(precio),1), count(*) as num_prop, row_number() over(order by avg(precio) desc) as ranking
from bootcamp.bronze.properties_bronze
group by zona



In [0]:
WITH promedios AS (
    SELECT
        zona,
        AVG(precio) OVER (PARTITION BY zona) AS promedio_zona,
        AVG(precio) OVER () AS promedio_total
    FROM bootcamp.bronze.properties_bronze
)
SELECT DISTINCT
    zona,
    promedio_zona,
    promedio_total,
    promedio_zona - promedio_total AS diferencia
FROM promedios;

In [0]:
--6.3
select 
    month(fecha::date) as mes,
    count(*) as num_prop,
    avg(precio) as precio_promedio
from bootcamp.bronze.properties_bronze
where fecha::date is not null
group by mes order by mes asc
;


In [0]:
--7.1
